[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/34_speculative_decoding.ipynb)

# 🔴 Hard: Speculative Decoding

Implement the **acceptance/rejection step** of speculative decoding — a technique for accelerating LLM inference.

### Signature
```python
def speculative_decode(target_probs, draft_probs, draft_tokens) -> list[int]:
    # target_probs: (K, V) from target (large) model
    # draft_probs: (K, V) from draft (small) model
    # draft_tokens: (K,) tokens sampled by draft model
    # Returns: list of accepted tokens (1 to K)
```

### Algorithm
For each position i = 0, ..., K-1:
1. `ratio = target_probs[i, token_i] / draft_probs[i, token_i]`
2. Accept with probability `min(1, ratio)`
3. If rejected: sample from `normalize(max(0, target - draft))`, append, and stop

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.1 MB/s eta 0:00:00


In [2]:

import torch

In [82]:
# ✏️ YOUR IMPLEMENTATION HERE

def speculative_decode(target_probs, draft_probs, draft_tokens):
    p_x = target_probs[torch.arange(target_probs.shape[0]), draft_tokens]
    q_x = draft_probs[torch.arange(target_probs.shape[0]), draft_tokens]
    a = torch.min(torch.ones(1), p_x/q_x)
    u = torch.distributions.uniform.Uniform(0, 1).rsample(a.shape)
    mask = u < a
    if mask.all():
      return draft_tokens
    first_mismatch = int(torch.argwhere(~mask)[0])
    accepted = draft_tokens[:first_mismatch]
    residual_prob = torch.maximum(torch.zeros(1), target_probs - draft_probs)
    residual_prob = residual_prob / residual_prob.sum(dim=-1, keepdim=True)
    output = torch.multinomial(residual_prob[first_mismatch], 1)
    output2 = torch.concat((accepted, output), 0)
    return output2
    pass  # accept/reject loop

In [86]:
def speculative_decode(target_probs, draft_probs, draft_tokens):
    rows = torch.arange(len(draft_tokens))
    p_x = target_probs[rows, draft_tokens]
    q_x = draft_probs[rows, draft_tokens]
    accept = torch.rand(len(draft_tokens)) < (p_x/q_x)
    if accept.all():
      return draft_tokens.tolist()
    first_mismatch = int(torch.argwhere(~accept)[0])
    residual = torch.clamp(target_probs[first_mismatch] - draft_probs[first_mismatch], min=0)
    residual = residual / residual.sum()
    resampled = torch.multinomial(residual, 1)
    return torch.cat((draft_tokens[:first_mismatch], resampled)).tolist()

In [87]:
# 🧪 Debug
torch.manual_seed(0)
probs = torch.softmax(torch.randn(4, 10), dim=-1)
tokens = torch.tensor([2, 5, 1, 8])
print('Perfect draft:', speculative_decode(probs, probs, tokens))
target = torch.softmax(torch.randn(4, 10), dim=-1)
draft = torch.softmax(torch.randn(4, 10), dim=-1)
print('Random draft:', speculative_decode(target, draft, tokens))

Perfect draft: [2, 5, 1, 8]
Random draft: [2, 5, 1, 8]


In [88]:
# 🧪 Better tests
import torch

def _run_tests():
    torch.manual_seed(0)
    V = 10

    # ---- Test 1: perfect draft accepts everything (deterministic) ----
    probs = torch.softmax(torch.randn(4, V), dim=-1)
    tokens = torch.tensor([2, 5, 1, 8])
    out = list(speculative_decode(probs, probs, tokens))
    assert len(out) == 4, f"perfect draft should accept all 4, got {len(out)}"
    assert out == tokens.tolist(), f"perfect draft should echo tokens, got {out}"
    print("✓ 1: perfect draft accepts all K")

    # ---- Test 2: certain rejection at position 0 ----
    # p_0(x_0) = 0  =>  ratio 0  =>  accept prob 0  =>  always reject at 0
    K, V = 3, 5
    target = torch.softmax(torch.randn(K, V), dim=-1)
    draft = torch.softmax(torch.randn(K, V), dim=-1)
    x0 = 2
    target[0, x0] = 0.0
    target[0] = target[0] / target[0].sum()          # renormalize
    toks = torch.tensor([x0, 0, 0])
    for _ in range(50):
        out = list(speculative_decode(target, draft, toks))
        assert len(out) == 1, f"forced reject at 0 => length 1, got {len(out)}"
        # resampled token must have support under the residual
        resid = torch.clamp(target[0] - draft[0], min=0)
        assert resid[out[0]] > 0, f"resampled token {out[0]} has zero residual mass"
    print("✓ 2: certain rejection returns single resampled token from residual")

    # ---- Test 3: residual is a valid distribution ----
    p = torch.softmax(torch.randn(V), dim=-1)
    q = torch.softmax(torch.randn(V), dim=-1)
    resid = torch.clamp(p - q, min=0)
    resid = resid / resid.sum()
    assert torch.all(resid >= 0), "residual has negative mass"
    assert abs(resid.sum().item() - 1.0) < 1e-5, "residual does not sum to 1"
    print("✓ 3: residual is non-negative and normalized")

    # ---- Test 4: DISTRIBUTIONAL GUARANTEE (the important one) ----
    # For K=1, the returned token must be distributed as target, regardless of draft.
    torch.manual_seed(1)
    V = 6
    p = torch.softmax(torch.randn(1, V), dim=-1)
    q = torch.softmax(torch.randn(1, V), dim=-1)
    N = 40000
    counts = torch.zeros(V)
    for _ in range(N):
        # draft samples its own token each trial
        x = torch.multinomial(q[0], 1)
        out = speculative_decode(p, q, x)
        counts[int(out[0])] += 1
    emp = counts / N
    max_err = (emp - p[0]).abs().max().item()
    assert max_err < 0.02, f"empirical dist deviates from target by {max_err:.4f}"
    print(f"✓ 4: output matches target distribution (max err {max_err:.4f})")

    print("\nAll tests passed.")

_run_tests()

✓ 1: perfect draft accepts all K
✓ 2: certain rejection returns single resampled token from residual
✓ 3: residual is non-negative and normalized
✓ 4: output matches target distribution (max err 0.0030)

All tests passed.


In [89]:
# ✅ SUBMIT
from torch_judge import check
check('speculative_decoding')


🧪 Testing: Speculative Decoding (Hard)
──────────────────────────────────────────────────
  ✅ [1/3] Perfect draft: all accepted (4.9ms)
  ✅ [2/3] Output length bounded (1.8ms)
  ✅ [3/3] All tokens valid (22.2ms)
──────────────────────────────────────────────────
  🎉 All 3 tests passed! (28.9ms total)
  Progress saved. Run status() to see your dashboard.

